In [1]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from downloading_the_base_model.download_model import downlaod_model

downlaod_model("Qwen/Qwen3-0.6B", "qwen")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/teamspace/studios/this_studio/open-posttraining-system/evaluating_reasoning_models/qwen'

In [2]:


import torch
from safetensors.torch import load_file

from base_model.qwen import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    Qwen3Tokenizer,
    generate_text,
    load_hf_weights_into_qwen,
)

model_dir = Path.cwd() / "qwen"

#def main(prompt):
def load_model_and_tokenizer(which_model, use_compile):

    if which_model == "base":
        tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json")
        
    elif which_model == "reasoning":
        tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json", apply_chat_template=True,   add_generation_prompt=True, add_thinking=True)
    
    

    else:
        raise ValueError(f"Not a valid model type")
    
    weights = load_file(model_dir / "model.safetensors")

    model = Qwen3Model(QWEN_CONFIG_06_B)
    load_hf_weights_into_qwen(
        model,
        param_config={
            "n_layers": QWEN_CONFIG_06_B["n_layers"],
            "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"],
        },
        params=weights,
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.to(torch.bfloat16)

    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)
     
    return model, tokenizer




In [3]:
### here
which_model = "base"


model, tokenizer = load_model_and_tokenizer(
    which_model=which_model,
    use_compile=False
)



In [4]:
import torch

from base_model.qwen import KVCache


device = "cuda" if torch.cuda.is_available() else "cpu"


@torch.inference_mode()
def generate_text_stream_with_kv_cache(prompt, model, tokenizer, device, max_new_tokens, eos_token_id):
    # Encode prompt and move to device
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

    # Set model to evaluation mode
    model.eval()

    # Initialize KV cache
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # Initial forward pass using full prompt
    logits = model(input_ids, cache=cache)[:, -1]

    # Autoregressive generation loop
    for _ in range(max_new_tokens):

        # Greedy decoding
        next_token = torch.argmax(logits,dim=-1,keepdim=True)

        # Stop if EOS token is generated
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break

        # Stream generated token
        yield next_token

        # Next forward pass only uses new token
        logits = model(next_token, cache=cache)[:, -1]

def generate_text(prompt, model, tokenizer, device, max_new_tokens, eos_token_id):


    for token in generate_text_stream_with_kv_cache(
        prompt=prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        eos_token_id=eos_token_id,
        device=device,
    ):
        output_tokens = token.squeeze(0).tolist()

        print(tokenizer.decode(output_tokens), end="", flush=True)

    
         

    


In [5]:
max_new_tokens = 600
device = "cuda" if torch.cuda.is_available() else "cpu"
eos_token_id = tokenizer.eos_token_id
prompt = ("If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?")

generate_text(prompt, model, tokenizer, device, max_new_tokens, eos_token_id)

 To solve this problem, we can use the identity that relates the sum and product of two numbers to their squares. The identity is:



$$
a^2 + b^2 = (a + b)^2 - 2ab
$$

Given that $a + b = 3$ and $ab = \frac{13}{6}$, we can substitute these values into the identity to find $a^2 + b^2$.

Let's compute this step by step:

1. Start with the given values:
   $$
   a + b = 3
   $$
   $$
   ab = \frac{13}{6}
   $$

2. Apply the identity:
   $$
   a^2 + b^2 = (a + b)^2 - 2ab
   $$

3. Substitute the given values:
   $$
   a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
   $$

4. Simplify the expression:
   $$
   a^2 + b^2 = 9 - \frac{26}{6}
   $$

5. Simplify the fraction:
   $$
   a^2 + b^2 = 9 - \frac{13}{3}
   $$

6. Convert 9 to a fraction with denominator 3:
   $$
   a^2 + b^2 = \frac{27}{3} - \frac{13}{3}
   $$

7. Subtract the fractions:
   $$
   a^2 + b^2 = \frac{14}{3}
   $$

So, the value of $a^2 + b^2$ is $\frac{14}{3}$.

Let me double-check the calculations to ensure there are no errors:

- $ (3)^2 = 9 $
- $ 2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3} $
- $ 9 - \frac{13}{3} = \frac{27}{3} - \frac{13}{3

In [6]:
from IPython.display import Math

display(Math(r"\dfrac{14}{3}"))

<IPython.core.display.Math object>

### creating a wrapper function

In [7]:
## ok writing the kv_cache token_stream generation and text_generation seperately
from base_model.qwen import KVCache
@torch.inference_mode()
def kvcache_based_token_generation(input_ids, model, max_new_tokens, eos_token_id):
    # initializing model in eval_model() 
    model.eval()   
    cache = KVCache(n_layers = model.cfg["n_layers"]) 
    model.reset_kv_cache()

    logits = model(input_ids, cache=cache)[:,-1]

    for _ in range(max_new_tokens):
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        if (eos_token_id is not None and 
            torch.all(next_token == eos_token_id)):
            break

        yield next_token

        logits = model(next_token, cache=cache)[:, -1]






## this function will use the existing kv_cache stream function---
def text_generation_wrapper(prompt, model, tokenizer, device, max_new_tokens, verbose=False):
    ## encode 
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)


    generated_ids = []
    for token in kvcache_based_token_generation(input_ids, model, max_new_tokens, tokenizer.eos_token_id):
        output_tokens = token.squeeze(0).tolist()
        ## append ids --scalar
        generated_ids.append(token.squeeze(0).item())
        if verbose:
            print(tokenizer.decode(output_tokens), end="", flush=True)

    return tokenizer.decode(generated_ids)


skip_portion = False

if not skip_portion:
    generated_text = text_generation_wrapper(prompt, model, tokenizer, device, max_new_tokens=600, verbose=True)
    




 To solve this problem, we can use the identity that relates the sum and product of two numbers to their squares. The identity is:

$$
a^2 + b^2 = (a

 + b)^2 - 2ab
$$

Given that $a + b = 3$ and $ab = \frac{13}{6}$, we can substitute these values into the identity to find $a^2 + b^2$.

Let's compute this step by step:

1. Start with the given values:
   $$
   a + b = 3
   $$
   $$
   ab = \frac{13}{6}
   $$

2. Apply the identity:
   $$
   a^2 + b^2 = (a + b)^2 - 2ab
   $$

3. Substitute the given values:
   $$
   a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
   $$

4. Simplify the expression:
   $$
   a^2 + b^2 = 9 - \frac{26}{6}
   $$

5. Simplify the fraction:
   $$
   a^2 + b^2 = 9 - \frac{13}{3}
   $$

6. Convert 9 to a fraction with denominator 3:
   $$
   a^2 + b^2 = \frac{27}{3} - \frac{13}{3}
   $$

7. Subtract the fractions:
   $$
   a^2 + b^2 = \frac{14}{3}
   $$

So, the value of $a^2 + b^2$ is $\frac{14}{3}$.

Let me double-check the calculations to ensure there are no errors:

- $ (3)^2 = 9 $
- $ 2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3} $
- $ 9 - \frac{13}{3} = \frac{27}{3} - \frac{13}{3} = \frac{14}{3} 

In [8]:
from IPython.display import Latex, display
display(Latex(generated_text))  

<IPython.core.display.Latex object>

In [9]:
generated_text

" To solve this problem, we can use the identity that relates the sum and product of two numbers to their squares. The identity is:\n\n$$\na^2 + b^2 = (a + b)^2 - 2ab\n$$\n\nGiven that $a + b = 3$ and $ab = \\frac{13}{6}$, we can substitute these values into the identity to find $a^2 + b^2$.\n\nLet's compute this step by step:\n\n1. Start with the given values:\n   $$\n   a + b = 3\n   $$\n   $$\n   ab = \\frac{13}{6}\n   $$\n\n2. Apply the identity:\n   $$\n   a^2 + b^2 = (a + b)^2 - 2ab\n   $$\n\n3. Substitute the given values:\n   $$\n   a^2 + b^2 = (3)^2 - 2 \\left( \\frac{13}{6} \\right)\n   $$\n\n4. Simplify the expression:\n   $$\n   a^2 + b^2 = 9 - \\frac{26}{6}\n   $$\n\n5. Simplify the fraction:\n   $$\n   a^2 + b^2 = 9 - \\frac{13}{3}\n   $$\n\n6. Convert 9 to a fraction with denominator 3:\n   $$\n   a^2 + b^2 = \\frac{27}{3} - \\frac{13}{3}\n   $$\n\n7. Subtract the fractions:\n   $$\n   a^2 + b^2 = \\frac{14}{3}\n   $$\n\nSo, the value of $a^2 + b^2$ is $\\frac{14}{3}$.\n

In [10]:
char = "to solve this problem, we use the identity that relates"

print(char.rfind("use"))

26


In [7]:
char1 = r"""
... some explanation...
**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]
"""


def get_last_boxed(text):
    boxed_start_idx = text.rfind(r"\boxed")

    if boxed_start_idx == -1:
        return None

    current_idx = boxed_start_idx + len(r"\boxed")

    # skip whitespace
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # must start with {
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    current_idx += 1
    brace_depth = 1
    content_start = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]

        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1

        current_idx += 1

    # unmatched braces
    if brace_depth != 0:
        return None

    return text[content_start:current_idx - 1]


print(get_last_boxed(char1))

\dfrac{14}{3}


In [3]:
import re

RE_NUMBER = re.compile(r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)")


def extract_final_candidate(text, fallback="number_then_full"):
    
    ## ok, so this is the default value--if nothing matched
    result = ""

    if text:
        ## last boxed, expression --- if present
        boxed = get_last_boxed(text.strip())
        if boxed:
            result = boxed.strip().strip("#,$ ") ## removing spaces, hashes and dollar signs

        elif fallback in ("number_then_full", "number_only"): ## if there was nothing in boxed, ---then fallback
            m = RE_NUMBER.findall(text)
            if m:
                ## use last number
                result = m[-1]
            elif fallback == "number_then_full":
                ## return full text, if no number found
                return text

    return result 
        

In [9]:
print(extract_final_candidate(char1))

\dfrac{14}{3}


In [10]:
print(extract_final_candidate(r"\dfrac{ 14/3 }"))


14/3


In [11]:
print(extract_final_candidate(r"\ abc 14/3 }"))


14/3


In [12]:
print(extract_final_candidate(r"hello <abc> not"))


hello <abc> not


In [4]:
import re
LATEX_FIXES = [
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
] 


RE_SPECIAL = re.compile(r"<\|[^>]+?\|>") ## skip the special tokens

def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()
    SUPERSCRIPT_MAP = {
        "⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",
        "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9",
        "⁺": "+", "⁻": "-", "⁽": "(", "⁾": ")",
    }


    match = re.match(r"^[A-Za-z]\s*[.:]\s*(.+)$", text) ## this will detect dots, simicolons before answer--and will  remove those
    if match:
        text = match.group(1) ## captured

    
    ## remove angle-degree markers
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text) ## {\circ}
    text = re.sub(r"\^\s*\\circ", "", text)
    text = text.replace("°", "")

    ## unwrap \text{this part will be captured} if the  whole string is wrapped 
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    
    #
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)


    ## latex canonicalization
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    def convert_superscripts(s, base=None):
        converted = "".join(
            SUPERSCRIPT_MAP[ch] if ch in SUPERSCRIPT_MAP else ch
            for ch in s
        )
        if base is None:
            return  converted
        return f"{base}**{converted}"
    
    ## convert unicode superscripts into exponent form 

    text = re.sub(
            r"([0-9A-Za-z\)\]\}])([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)",
            lambda m: convert_superscripts(m.group(2), base=m.group(1)),
            text,
    )

    text = convert_superscripts(text)

    ## numbers/roots
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
            r"\\sqrt\s*\{([^}]*)\}",
            lambda match: f"sqrt({match.group(1)})",
            text,
        )
    
    # numbers/roots
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # fractions
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # exponent and mixed numbers
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # 1,234 -> 1234
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    return text.replace("{", "").replace("}", "").strip().lower()

In [14]:
print(normalize_text(extract_final_candidate(char1)))

(14)/(3)


In [15]:
print(normalize_text(r"$\dfrac{14}{3.}$"))

(14)/(3.)


In [16]:
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))

(14)/(3)


In [17]:
print(normalize_text("4/3"))

4/3


### here, will see if the answer generated by llm, matches the "ground_truth"

In [5]:
import sympy.parsing.sympy_parser as spp
from sympy.core.sympify import SympifyError
from tokenize import TokenError
from sympy.polys.polyerrors import PolynomialError

def sympy_parser(expr):

    # 1. Guard rails against memory exhaustion and null inputs--badly trained models generate some --garbage
    if not expr or len(expr) > 2000:
        return None

    try:
        # 2. Parse and return the evaluated expression
        return spp.parse_expr(
            expr,
            transformations=(
                *spp.standard_transformations,  ##stardard  transformation for handling parenthesis
                spp.implicit_multiplication_application, ## aloweing omitted  multiplication  symbols
            ),
            evaluate=True
        )
    except (
        SympifyError, 
        SyntaxError, 
        TypeError, 
        AttributeError, 
        IndexError, 
        TokenError, 
        ValueError, 
        PolynomialError,
        ZeroDivisionError  # Added to catch literal divisions like '1/0' during evaluation
    ):
        return None

In [19]:
print(sympy_parser(normalize_text(extract_final_candidate(char1))))

14/3


In [20]:
print(sympy_parser("28/6"))

14/3


In [6]:
from sympy import simplify

def equality_check(expr_gtruth, expr_pred):
    ## checking if both of the expressions are, exactly the same string
    if expr_gtruth ==  expr_pred:
        return True

    ## parsing both expressions into sympy parser--- returns None if parsing fails
    gtruth, pred = sympy_parser(expr_gtruth), sympy_parser(expr_pred)


    ## if both expression were parsed successfully, now try---> symbolic  comparison
    if gtruth is not None and pred is not None:
        try:
        ## here if the difference is 0, they are equivalent
            return simplify(gtruth - pred) == 0

        except (SympifyError, TypeError):
            pass
    
    return  False




In [22]:
print(equality_check(normalize_text("13/4."), normalize_text(r"(13)/(4)")))

True


In [23]:
print(equality_check(normalize_text("13/5."), normalize_text(r"(13)/(4)")))

False


In [24]:
print(equality_check(normalize_text("0.5"),normalize_text(r"(1)/(2)")))

True


### now, grading answers

In [7]:

def split_into_parts(text):
    result = [text]

    if text:
        ##  check if text looks like a tuple of list---e.g "(a,b)"  or "[a,b]"
        if (len(text) >= 2 and text[0] in "([" and text[-1] in ")]" and "," in text[1:-1]):
            ## split on comma
            items = [i.strip() for i in text[1:-1].split(",")]
            if all(items):
                result = items 

    else:
        ## text is empty, so return the empty list
        result = []

    return result 

In [26]:
print(normalize_text(r"(14/3, 12/2)"))

(14/3, 12/2)


In [8]:
def grade_answer(pd_text, gdt_text):
    result = False   ## this will be the default outcome,,--if the checks fail here

    ## only continuing if both of the inputs  are non-empty strings
    if pd_text is not None and gdt_text is not None:
        gdt_parts = split_into_parts(normalize_text(gdt_text)) ##  breaking gdt_text ---> into comparable parts

        pd_parts  = split_into_parts(normalize_text(pd_text)) ## braking  prediction text ----> into comparable parts


        ## ensuring both sides --have same number of valid parts
        if (gdt_parts and pd_parts and len(gdt_parts) == len(pd_parts)):
            result = all(equality_check(gt, pred) for gt, pred in zip(gdt_parts, pd_parts))


    return result  ## true only when all the checks are passed..

In [28]:
grade_answer("14/3", r"\frac{14}{3}")

True

In [29]:
grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)")

True

In [15]:
## defining the test cases:  (name, prediction, ground truth, expected_truth)

tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
        ("check_16", r"2²", "2**2", True),
    ]






def run_demos_table(tests):
    header = ("Test", "Expected","Got", "Status")
    rows = []
    for name, prediction, ground_truth, expected_truth in tests:

        got = grade_answer(prediction, ground_truth)
        status = "PASS" if got == expected_truth else "FAIL"

        rows.append((name, str(expected_truth),str(got),status)) ##3 this will show --> the name of the test, the expectation we had, the real output we got after performing the grade_answer check, and the status if that was passed or failed


    data = [header] + rows


    ## getting the max_column width
    col_widths = [
            max(len(row[i]) for row in data) for i in range(len(header))
        ]    
    ## print table row by row
    for row in data:
        line = " | ".join(row[i].ljust(col_widths[i]) for i in range(len(header)))
        print(line)


    # Print summary of passed tests
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")


In [16]:
run_demos_table(tests)

Test     | Expected | Got   | Status
check_1  | True     | True  | PASS  
check_2  | True     | True  | PASS  
check_3  | True     | True  | PASS  
check_4  | True     | True  | PASS  
check_5  | True     | True  | PASS  
check_6  | False    | False | PASS  
check_7  | False    | False | PASS  
check_8  | True     | True  | PASS  
check_9  | False    | False | PASS  
check_10 | True     | True  | PASS  
check_11 | True     | True  | PASS  
check_12 | False    | False | PASS  
check_13 | True     | True  | PASS  
check_14 | True     | True  | PASS  
check_15 | True     | True  | PASS  
check_16 | True     | True  | PASS  

Passed 16/16
